# Create Junta de Andalucía Awards

Creates Junta de Andalucía research/R&D+i awards from the Junta's open-data bulk subsidies dataset.

**Prerequisites:** run `scripts/local/andalucia_to_s3.py` to download + filter + aggregate + upload first.

**Data source:** "Subvenciones otorgadas por la Junta de Andalucía" on the Junta CKAN portal (juntadeandalucia.es/datosabiertos); bulk file via `https://datos.juntadeandalucia.es/api/v0/subventions/all?format=csv` (~4.5M subsidy rows since ~2016, pipe-delimited).

**INCLUSION RULE (research scoping, applied in the script):** keep a row iff (a) budget programme `program = '54A'` ("INVESTIGACIÓN CIENTÍFICA E INNOVACIÓN": PAIDI projects/groups, predoctoral & young-researcher contracts, R&D infrastructure, Severo Ochoa/María de Maeztu co-funding), OR (b) announcement/finality/regulatory-base text matches the research regex (investigación, I+D+i, científic-, PAIDI, pre/postdoctoral, Talentia) — pulling in biomedical research (41K), business R&D (72A/72B), and nominative research subsidies (CSIC Doñana, royal academies) — minus the exclusion regex (educational innovation 54C, employment training, commercial innovation).

**S3 location:** `s3a://openalex-ingest/awards/andalucia/andalucia_projects.parquet`

**Junta de Andalucía funder in OpenAlex:** funder_id 4320326754 · display_name "Junta de Andalucía" · ROR https://ror.org/01jem9c82 · doi 10.13039/501100011011 · ES.

**Schema notes:**
- **Amounts are EUR whole units** (`amount` column, decimal point).
- `funder_award_id` = `{id_system_internal}:{beneficiary}` — the source has no per-grant id; the grant-resolution batch id + verbatim beneficiary string is the natural key, aggregated in the script (sum amount, min grant_date). The dataset's `id_seq` is NOT used (row sequence, renumbered on file regeneration).
- Beneficiaries: **institutions** (universities, CSIC institutes, companies) ship as `lead_investigator.affiliation.name`; **physical persons** (fellows; `physical_person = 'X'`) arrive as "GIVEN SURNAME1 SURNAME2" with no comma, so a reliable given/family split is impossible — `lead_investigator` is NULL for person rows (raw name preserved in the raw table).
- `grant_date` = concession date (single-dated register; no end dates).

provenance `andalucia`, priority 421.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.andalucia_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/andalucia/andalucia_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.andalucia_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.andalucia_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.andalucia_raw LIMIT 5;

## Step 1.6: Funder existence fail-fast

Must return exactly 1 row (F4320326754 is Crossref-registered / Path A). If 0, STOP.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320326754;


## Step 2: Create Andalucía Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.andalucia_awards
USING delta
AS
WITH
andalucia_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320326754  -- Junta de Andalucía
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.announcement as display_name,
        NULLIF(TRIM(g.finality), '') as description,
        f.funder_id,
        g.funder_award_id,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'EUR' ELSE NULL END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        CASE
            WHEN LOWER(g.announcement) RLIKE '(predoctoral|postdoctoral|posdoctoral|contrat|talentia|beca)' THEN 'fellowship'
            ELSE 'research'
        END as funding_type,
        NULLIF(TRIM(g.name_program), '') as funder_scheme,
        'andalucia' as provenance,
        TRY_TO_DATE(g.grant_date, 'yyyy-MM-dd') as start_date,
        CAST(NULL AS DATE) as end_date,
        COALESCE(YEAR(TRY_TO_DATE(g.grant_date, 'yyyy-MM-dd')), TRY_CAST(g.award_year AS INT)) as start_year,
        CAST(NULL AS INT) as end_year,
        -- Institutions -> affiliation; physical persons -> NULL (name format
        -- "GIVEN SURNAME1 SURNAME2" cannot be split reliably, see header).
        CASE
            WHEN g.institution_name IS NOT NULL AND TRIM(g.institution_name) != '' THEN
                struct(
                    CAST(NULL AS STRING) as given_name,
                    CAST(NULL AS STRING) as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution_name as name,
                        'Spain' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        'https://www.juntadeandalucia.es/datosabiertos/portal/dataset/subvenciones-otorgadas-por-la-junta-de-andalucia' as landing_page_url,
        CAST(NULL AS STRING) as doi
    FROM openalex.awards.andalucia_raw g
    CROSS JOIN andalucia_funder f
)
SELECT *,
    concat('https://api.openalex.org/works?filter=awards.id:G', id) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM awards_transformed;


## Step 3: Insert into openalex_awards_raw (priority 421)

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'andalucia' AND priority = 421;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    421 as priority
FROM openalex.awards.andalucia_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_andalucia_awards FROM openalex.awards.andalucia_awards;

In [ ]:
%sql
SELECT funder_award_id, display_name, funding_type, amount, currency, start_year,
       lead_investigator.affiliation.name
FROM openalex.awards.andalucia_awards LIMIT 10;


In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt FROM openalex.awards.andalucia_awards
GROUP BY funding_type ORDER BY cnt DESC;


In [ ]:
%sql
-- §6.3 completeness. NOTE: pct_with_pi expected 0% — the source publishes no
-- PI names (institutional grants) and person-beneficiary names cannot be
-- split reliably (see header). pct_with_amount expected >95%.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(lead_investigator.family_name) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(start_date) as has_start_date,
    ROUND(try_divide(COUNT(amount) * 100.0, COUNT(*)), 1) as pct_with_amount,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name) * 100.0, COUNT(*)), 1) as pct_with_institution
FROM openalex.awards.andalucia_awards;


In [ ]:
%sql
-- §6.7 amount/currency coverage (FAIL-FAST)
SELECT COUNT(*) AS total, COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies, collect_set(currency) AS currencies,
    MIN(amount) AS min_amount, MAX(amount) AS max_amount, AVG(amount) AS avg_amount
FROM openalex.awards.andalucia_awards;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.andalucia_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year DESC LIMIT 20;


In [ ]:
%sql
SELECT lead_investigator.affiliation.name as institution, COUNT(*) as grant_count
FROM openalex.awards.andalucia_awards WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY 1 ORDER BY 2 DESC LIMIT 20;
